# 🏥 Dashboard – Análisis de Obesidad
**Autor:** Sergio Roa  
**Curso:** Ingeniería en Ciencia de Datos | 2026

---


In [ ]:
# Celda 1 – Instalación de dependencias
import sys
!{sys.executable} -m pip install dash plotly scikit-learn scipy jupyter-dash -q
print('✅ Dependencias instaladas correctamente')

In [ ]:
# Celda 2 – Dashboard completo
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy import stats

# ── Dataset ──────────────────────────────────────────────────
np.random.seed(42)
n = 500
edades     = np.random.normal(24, 6, n).clip(14, 61).astype(int)
alturas    = np.random.normal(1.70, 0.10, n).clip(1.45, 1.98).round(2)
pesos      = np.random.normal(86, 26, n).clip(39, 173).round(1)
imc        = (pesos / (alturas ** 2)).round(2)
act_fisica = np.random.choice([0,1,2,3], n, p=[0.25,0.35,0.28,0.12])
cons_agua  = np.random.choice([1,2,3],   n, p=[0.25,0.45,0.30])
com_cal    = np.random.choice([0,1],     n, p=[0.40,0.60])
uso_tec    = np.random.choice([0,1,2],   n, p=[0.20,0.45,0.35])

def nivel_obesidad(v):
    if v < 18.5: return 'Bajo peso'
    elif v < 25: return 'Normal'
    elif v < 30: return 'Sobrepeso'
    elif v < 35: return 'Obesidad I'
    elif v < 40: return 'Obesidad II'
    else:        return 'Obesidad III'

niveles = [nivel_obesidad(v) for v in imc]
obs_bin = (imc >= 30).astype(int)

df = pd.DataFrame({
    'Edad':edades,'Altura':alturas,'Peso':pesos,'IMC':imc,
    'Actividad_Fisica':act_fisica,'Consumo_Agua':cons_agua,
    'Comida_Calorica':com_cal,'Uso_Tecnologia':uso_tec,
    'Nivel_Obesidad':niveles,'Obesidad_Binaria':obs_bin
})

# ── Modelos ───────────────────────────────────────────────────
X_lin = df[['Peso','Edad','Actividad_Fisica','Consumo_Agua']].values
y_lin = df['IMC'].values
X_tr,X_te,y_tr,y_te = train_test_split(X_lin,y_lin,test_size=0.3,random_state=42)
mlr = LinearRegression().fit(X_tr,y_tr)
y_pred_lr = mlr.predict(X_te)
r2  = r2_score(y_te,y_pred_lr)
mse = mean_squared_error(y_te,y_pred_lr)

X_log = df[['Peso','Edad','Actividad_Fisica','Consumo_Agua','Comida_Calorica']].values
y_log = df['Obesidad_Binaria'].values
scaler   = StandardScaler()
X_log_sc = scaler.fit_transform(X_log)
Xl_tr,Xl_te,yl_tr,yl_te = train_test_split(X_log_sc,y_log,test_size=0.3,random_state=42,stratify=y_log)
mlog = LogisticRegression(C=1,solver='lbfgs',max_iter=1000,random_state=42).fit(Xl_tr,yl_tr)
y_pred_log = mlog.predict(Xl_te)
acc = mlog.score(Xl_te,yl_te)
cm  = confusion_matrix(yl_te,y_pred_log)

activos     = df[df['Actividad_Fisica']>=2]['IMC'].values
sedentarios = df[df['Actividad_Fisica']<=1]['IMC'].values
t_stat,p_val = stats.ttest_ind(activos,sedentarios)
p_uni = p_val/2

# ── Estilos ───────────────────────────────────────────────────
ORDEN  = ['Bajo peso','Normal','Sobrepeso','Obesidad I','Obesidad II','Obesidad III']
COLORES= {'Bajo peso':'#4CAF50','Normal':'#8BC34A','Sobrepeso':'#FFC107',
           'Obesidad I':'#FF9800','Obesidad II':'#F44336','Obesidad III':'#9C27B0'}
CARD   = {'backgroundColor':'#fff','borderRadius':'12px','padding':'20px',
           'marginBottom':'20px','boxShadow':'0 2px 8px rgba(0,0,0,0.08)'}
MET    = {**CARD,'textAlign':'center','padding':'16px 10px'}

# ── App ───────────────────────────────────────────────────────
app = Dash(__name__)
app.title = 'Dashboard Obesidad – Sergio Roa'

app.layout = html.Div([

    # Header
    html.Div([
        html.H1('🏥 Análisis de Factores Asociados a la Obesidad',
                style={'color':'white','margin':'0','fontSize':'22px'}),
        html.P('Ingeniería en Ciencia de Datos | Sergio Roa | 2026',
               style={'color':'#BBDEFB','margin':'4px 0 0 0','fontSize':'12px'})
    ], style={'backgroundColor':'#1565C0','padding':'20px 30px','marginBottom':'20px'}),

    # Métricas
    html.Div([
        html.Div([html.H3(f'{len(df)}',style={'color':'#1565C0','margin':'0','fontSize':'28px'}),
                  html.P('Pacientes',style={'color':'#666','margin':'4px 0 0 0','fontSize':'12px'})],
                 style={**MET,'flex':'1','margin':'0 6px'}),
        html.Div([html.H3(f'{df["IMC"].mean():.1f}',style={'color':'#E53935','margin':'0','fontSize':'28px'}),
                  html.P('IMC Promedio',style={'color':'#666','margin':'4px 0 0 0','fontSize':'12px'})],
                 style={**MET,'flex':'1','margin':'0 6px'}),
        html.Div([html.H3(f'{r2:.3f}',style={'color':'#2E7D32','margin':'0','fontSize':'28px'}),
                  html.P('R² Reg. Lineal',style={'color':'#666','margin':'4px 0 0 0','fontSize':'12px'})],
                 style={**MET,'flex':'1','margin':'0 6px'}),
        html.Div([html.H3(f'{acc*100:.1f}%',style={'color':'#6A1B9A','margin':'0','fontSize':'28px'}),
                  html.P('Accuracy Logística',style={'color':'#666','margin':'4px 0 0 0','fontSize':'12px'})],
                 style={**MET,'flex':'1','margin':'0 6px'}),
        html.Div([html.H3(f'{p_uni:.4f}',style={'color':'#E65100','margin':'0','fontSize':'28px'}),
                  html.P('p-valor Hipótesis',style={'color':'#666','margin':'4px 0 0 0','fontSize':'12px'})],
                 style={**MET,'flex':'1','margin':'0 6px'}),
    ], style={'display':'flex','padding':'0 20px','marginBottom':'10px'}),

    # Tabs
    dcc.Tabs([

        # TAB 1
        dcc.Tab(label='📊 Exploración', children=[
            html.Div([
                html.Div([
                    html.Div([
                        html.Label('Nivel de obesidad:', style={'fontWeight':'bold','fontSize':'13px'}),
                        dcc.Dropdown(id='fil-nivel',
                            options=[{'label':'Todos','value':'Todos'}]+[{'label':n,'value':n} for n in ORDEN],
                            value='Todos', clearable=False, style={'marginTop':'6px'})
                    ], style={'flex':'1','marginRight':'20px'}),
                    html.Div([
                        html.Label('Variable:', style={'fontWeight':'bold','fontSize':'13px'}),
                        dcc.Dropdown(id='fil-var',
                            options=[{'label':'IMC','value':'IMC'},{'label':'Peso','value':'Peso'},
                                     {'label':'Edad','value':'Edad'},{'label':'Altura','value':'Altura'}],
                            value='IMC', clearable=False, style={'marginTop':'6px'})
                    ], style={'flex':'1'})
                ], style={**CARD,'display':'flex'}),
                html.Div([
                    html.Div([dcc.Graph(id='hist')], style={**CARD,'flex':'1','marginRight':'12px'}),
                    html.Div([dcc.Graph(id='bar')],  style={**CARD,'flex':'1'})
                ], style={'display':'flex'}),
                html.Div([dcc.Graph(id='box')], style=CARD),
            ], style={'padding':'20px'})
        ]),

        # TAB 2
        dcc.Tab(label='🔬 Hipótesis', children=[
            html.Div([
                html.Div([
                    html.H3('Contraste de Hipótesis – Prueba t de Student',style={'color':'#1565C0'}),
                    html.Div([
                        html.Div([html.H4('H₀',style={'color':'#555'}),
                                  html.P('No existe diferencia significativa en el IMC entre activos y sedentarios.')],
                                 style={**CARD,'flex':'1','marginRight':'12px','borderLeft':'4px solid #90A4AE'}),
                        html.Div([html.H4('H₁',style={'color':'#E53935'}),
                                  html.P('Los activos tienen IMC significativamente menor que los sedentarios.')],
                                 style={**CARD,'flex':'1','borderLeft':'4px solid #E53935'}),
                    ], style={'display':'flex'}),
                    html.Div([
                        html.Div([html.H4(f't = {t_stat:.4f}',style={'color':'#1565C0','margin':'0'}),
                                  html.P('Estadístico t',style={'color':'#666','margin':'4px 0 0 0'})],
                                 style={**MET,'flex':'1','margin':'0 6px'}),
                        html.Div([html.H4(f'p = {p_uni:.4f}',style={'color':'#E53935','margin':'0'}),
                                  html.P('p-valor unilateral',style={'color':'#666','margin':'4px 0 0 0'})],
                                 style={**MET,'flex':'1','margin':'0 6px'}),
                        html.Div([html.H4('α = 0.05',style={'color':'#2E7D32','margin':'0'}),
                                  html.P('Significancia',style={'color':'#666','margin':'4px 0 0 0'})],
                                 style={**MET,'flex':'1','margin':'0 6px'}),
                        html.Div([html.H4('✅ Se rechaza H₀' if p_uni<0.05 else '❌ No se rechaza H₀',
                                          style={'color':'#2E7D32' if p_uni<0.05 else '#E53935','margin':'0','fontSize':'14px'}),
                                  html.P('Decisión',style={'color':'#666','margin':'4px 0 0 0'})],
                                 style={**MET,'flex':'1','margin':'0 6px'}),
                    ], style={'display':'flex','marginBottom':'16px'}),
                ], style=CARD),
                html.Div([dcc.Graph(id='graf-hip')], style=CARD),
            ], style={'padding':'20px'})
        ]),

        # TAB 3
        dcc.Tab(label='🤖 Modelos', children=[
            html.Div([
                html.Div([
                    html.H3('Regresión Lineal Múltiple',style={'color':'#1565C0'}),
                    html.Div([
                        html.Div([html.H4(f'R² = {r2:.4f}',style={'color':'#2E7D32','margin':'0'}),
                                  html.P('Coef. determinación',style={'color':'#666','margin':'4px 0 0 0'})],
                                 style={**MET,'flex':'1','margin':'0 6px'}),
                        html.Div([html.H4(f'MSE = {mse:.4f}',style={'color':'#E53935','margin':'0'}),
                                  html.P('Error cuadrático',style={'color':'#666','margin':'4px 0 0 0'})],
                                 style={**MET,'flex':'1','margin':'0 6px'}),
                        html.Div([html.H4(f'RMSE = {np.sqrt(mse):.4f}',style={'color':'#1565C0','margin':'0'}),
                                  html.P('Raíz del MSE',style={'color':'#666','margin':'4px 0 0 0'})],
                                 style={**MET,'flex':'1','margin':'0 6px'}),
                    ], style={'display':'flex','marginBottom':'16px'}),
                    dcc.Graph(id='graf-rl'),
                ], style=CARD),
                html.Div([
                    html.H3('Regresión Logística',style={'color':'#1565C0'}),
                    html.Div([
                        html.Div([html.H4(f'{acc*100:.1f}%',style={'color':'#2E7D32','margin':'0'}),
                                  html.P('Accuracy',style={'color':'#666','margin':'4px 0 0 0'})],
                                 style={**MET,'flex':'1','margin':'0 6px'}),
                        html.Div([html.H4(f'{cm[1,1]/(cm[1,0]+cm[1,1])*100:.1f}%',style={'color':'#1565C0','margin':'0'}),
                                  html.P('Sensibilidad',style={'color':'#666','margin':'4px 0 0 0'})],
                                 style={**MET,'flex':'1','margin':'0 6px'}),
                        html.Div([html.H4(f'{cm[0,0]/(cm[0,0]+cm[0,1])*100:.1f}%',style={'color':'#6A1B9A','margin':'0'}),
                                  html.P('Especificidad',style={'color':'#666','margin':'4px 0 0 0'})],
                                 style={**MET,'flex':'1','margin':'0 6px'}),
                    ], style={'display':'flex','marginBottom':'16px'}),
                    dcc.Graph(id='graf-cm'),
                ], style=CARD),
            ], style={'padding':'20px'})
        ]),

        # TAB 4
        dcc.Tab(label='🔮 Predictor', children=[
            html.Div([
                html.Div([
                    html.H3('Predice el IMC de un paciente',style={'color':'#1565C0'}),
                    html.P('Ajusta los valores y obtén el IMC estimado y nivel de riesgo.'),
                    html.Label('Peso (kg)',style={'fontWeight':'bold'}),
                    dcc.Slider(id='sl-peso',min=40,max=170,step=1,value=80,
                               marks={40:'40',80:'80',120:'120',170:'170'},
                               tooltip={'placement':'bottom','always_visible':True}),
                    html.Br(),
                    html.Label('Edad (años)',style={'fontWeight':'bold'}),
                    dcc.Slider(id='sl-edad',min=14,max=61,step=1,value=25,
                               marks={14:'14',25:'25',40:'40',61:'61'},
                               tooltip={'placement':'bottom','always_visible':True}),
                    html.Br(),
                    html.Label('Días de actividad física/semana',style={'fontWeight':'bold'}),
                    dcc.Slider(id='sl-act',min=0,max=3,step=1,value=1,
                               marks={0:'0',1:'1',2:'2',3:'3'},
                               tooltip={'placement':'bottom','always_visible':True}),
                    html.Br(),
                    html.Label('Consumo de agua (litros/día)',style={'fontWeight':'bold'}),
                    dcc.Slider(id='sl-agua',min=1,max=3,step=1,value=2,
                               marks={1:'1L',2:'2L',3:'3L'},
                               tooltip={'placement':'bottom','always_visible':True}),
                    html.Div(id='pred-result',style={'marginTop':'24px'})
                ], style=CARD)
            ], style={'padding':'20px'})
        ]),

        # TAB 5
        dcc.Tab(label='📝 Conclusiones', children=[
            html.Div([
                html.Div([
                    html.H3('Conclusiones del Proyecto',style={'color':'#1565C0'}),
                    html.Hr(),
                    html.H4('1. Sobre el Problema'),
                    html.P('Más del 50% de los pacientes presentan algún grado de sobrepeso u obesidad, '
                           'consistente con las estadísticas epidemiológicas en América Latina.'),
                    html.H4('2. Contraste de Hipótesis'),
                    html.P(f'La prueba t arrojó p = {p_uni:.4f} < 0.05, confirmando que las personas '
                           'activas tienen un IMC significativamente menor que las sedentarias.'),
                    html.H4('3. Regresión Lineal Múltiple'),
                    html.P(f'R² = {r2:.4f}. El peso fue el predictor más fuerte del IMC. '
                           'La actividad física mostró un efecto protector (coeficiente negativo).'),
                    html.H4('4. Regresión Logística'),
                    html.P(f'Accuracy = {acc*100:.1f}%. El modelo clasifica correctamente '
                           'la mayoría de los casos de obesidad.'),
                    html.H4('5. Recomendaciones Clínicas'),
                    html.Ul([
                        html.Li('Incluir planes de actividad física en el tratamiento estándar.'),
                        html.Li('Monitorear el consumo de agua en los pacientes.'),
                        html.Li('Usar modelos predictivos como apoyo en el diagnóstico inicial.'),
                        html.Li('Recopilar más variables clínicas para mejorar el modelo.'),
                    ]),
                ], style=CARD)
            ], style={'padding':'20px'})
        ]),

    ], style={'fontFamily':'Segoe UI, sans-serif'})

], style={'backgroundColor':'#F5F7FA','minHeight':'100vh','fontFamily':'Segoe UI, sans-serif'})

# ── Callbacks ─────────────────────────────────────────────────
@app.callback([Output('hist','figure'),Output('bar','figure'),Output('box','figure')],
              [Input('fil-nivel','value'),Input('fil-var','value')])
def update_exp(nivel, var):
    dff = df if nivel=='Todos' else df[df['Nivel_Obesidad']==nivel]
    h = px.histogram(dff,x=var,nbins=30,color='Nivel_Obesidad',
                     color_discrete_map=COLORES,category_orders={'Nivel_Obesidad':ORDEN},
                     title=f'Distribución de {var}',template='plotly_white')
    h.update_layout(margin=dict(t=40,b=20,l=20,r=20))
    c = df['Nivel_Obesidad'].value_counts().reindex(ORDEN,fill_value=0).reset_index()
    c.columns=['Nivel','Cantidad']
    b = px.bar(c,x='Nivel',y='Cantidad',color='Nivel',color_discrete_map=COLORES,
               title='Pacientes por nivel',template='plotly_white',text='Cantidad')
    b.update_traces(textposition='outside')
    b.update_layout(showlegend=False,margin=dict(t=40,b=20,l=20,r=20))
    et = {0:'Ninguna',1:'1-2 días',2:'2-4 días',3:'4-5 días'}
    dff2 = dff.copy(); dff2['Act_Label']=dff2['Actividad_Fisica'].map(et)
    bx = px.box(dff2,x='Act_Label',y='IMC',color='Act_Label',
                category_orders={'Act_Label':['Ninguna','1-2 días','2-4 días','4-5 días']},
                title='IMC según actividad física',template='plotly_white')
    bx.update_layout(showlegend=False,margin=dict(t=40,b=20,l=20,r=20))
    return h, b, bx

@app.callback(Output('graf-hip','figure'), Input('fil-nivel','value'))
def update_hip(_):
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=activos,name=f'Activos (μ={np.mean(activos):.1f})',
                               marker_color='#42A5F5',opacity=0.7,nbinsx=25))
    fig.add_trace(go.Histogram(x=sedentarios,name=f'Sedentarios (μ={np.mean(sedentarios):.1f})',
                               marker_color='#EF5350',opacity=0.7,nbinsx=25))
    fig.add_vline(x=np.mean(activos),line_dash='dash',line_color='#1565C0',
                  annotation_text=f'Media activos: {np.mean(activos):.1f}')
    fig.add_vline(x=np.mean(sedentarios),line_dash='dash',line_color='#B71C1C',
                  annotation_text=f'Media sedentarios: {np.mean(sedentarios):.1f}')
    fig.update_layout(barmode='overlay',template='plotly_white',
                      title='IMC: Activos vs Sedentarios',
                      xaxis_title='IMC (kg/m²)',yaxis_title='Frecuencia',
                      margin=dict(t=50,b=20,l=20,r=20))
    return fig

@app.callback([Output('graf-rl','figure'),Output('graf-cm','figure')],
              Input('fil-nivel','value'))
def update_modelos(_):
    lims = [min(y_te.min(),y_pred_lr.min())-2, max(y_te.max(),y_pred_lr.max())+2]
    rl = go.Figure()
    rl.add_trace(go.Scatter(x=y_te,y=y_pred_lr,mode='markers',
                             marker=dict(color='#42A5F5',opacity=0.6,size=6),name='Pacientes'))
    rl.add_trace(go.Scatter(x=lims,y=lims,mode='lines',
                             line=dict(color='red',dash='dash',width=2),name='Predicción perfecta'))
    rl.update_layout(template='plotly_white',title=f'Reales vs Predichos | R²={r2:.3f}',
                     xaxis_title='IMC Real',yaxis_title='IMC Predicho',
                     margin=dict(t=50,b=20,l=20,r=20))
    etq = ['Sin obesidad','Con obesidad']
    cm_fig = px.imshow(cm,text_auto=True,x=etq,y=etq,color_continuous_scale='Blues',
                       title='Matriz de Confusión',
                       labels=dict(x='Predicción',y='Valor Real'))
    cm_fig.update_layout(template='plotly_white',margin=dict(t=50,b=20,l=20,r=20))
    return rl, cm_fig

@app.callback(Output('pred-result','children'),
              [Input('sl-peso','value'),Input('sl-edad','value'),
               Input('sl-act','value'),Input('sl-agua','value')])
def update_pred(peso,edad,act,agua):
    imc_p = mlr.predict([[peso,edad,act,agua]])[0]
    niv   = nivel_obesidad(imc_p)
    col   = COLORES.get(niv,'#333')
    prob  = mlog.predict_proba(scaler.transform([[peso,edad,act,agua,0]]))[0][1]
    return html.Div([
        html.H2(f'IMC Estimado: {imc_p:.2f} kg/m²',style={'color':col,'margin':'0'}),
        html.H3(f'Clasificación: {niv}',style={'color':col,'margin':'8px 0 0 0'}),
        html.P(f'Probabilidad de obesidad: {prob*100:.1f}%',style={'fontSize':'16px','marginTop':'12px'}),
        html.Hr(),
        html.P('Estimación basada en el modelo de regresión lineal múltiple.',
               style={'color':'#666','fontSize':'12px'})
    ], style={**CARD,'borderLeft':f'6px solid {col}'})

# ── Lanzar ────────────────────────────────────────────────────
app.run(jupyter_mode='inline', debug=False, port=8050)
